In [2]:

from pathlib import Path
import zipfile
import torch

# Project directories
PROJECT_DIR = Path(r"E:\punctuation_project")
DATA_DIR = PROJECT_DIR / "data"
MODELS_DIR = PROJECT_DIR / "models"

# Saved model locations
PUNCTUATION_MODEL_DIR = MODELS_DIR / "indicbert_punctuation_weighted_final"
TRANSLATION_MODEL_DIR = MODELS_DIR / "hindi_english_translation"

TRANSLATION_CHECKPOINT = (
    TRANSLATION_MODEL_DIR / "epoch5_checkpoint.pt"
)

TRANSLATION_BACKUP = (
    TRANSLATION_MODEL_DIR / "epoch5_final_backup.pt"
)

# Input ZIP
ZIP_PATH = DATA_DIR / "Sample-Raw-No-Punctuations.zip"

print("PROJECT CHECK")
print("=" * 50)

print("Project exists:", PROJECT_DIR.exists())
print("Data directory exists:", DATA_DIR.exists())
print("Models directory exists:", MODELS_DIR.exists())

print("\nPUNCTUATION MODEL")
print("Directory exists:", PUNCTUATION_MODEL_DIR.exists())

if PUNCTUATION_MODEL_DIR.exists():
    print("Files:", [p.name for p in PUNCTUATION_MODEL_DIR.iterdir()])

print("\nTRANSLATION MODEL")
print("Directory exists:", TRANSLATION_MODEL_DIR.exists())
print("Epoch 5 checkpoint:", TRANSLATION_CHECKPOINT.exists())
print("Epoch 5 backup:", TRANSLATION_BACKUP.exists())

print("\nINPUT ZIP")
print("ZIP exists:", ZIP_PATH.exists())

if ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        print("Files inside ZIP:")
        for name in z.namelist():
            print(" ", name)

print("\nGPU")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / (1024**3), 2
    ), "GB")


PROJECT CHECK
Project exists: True
Data directory exists: True
Models directory exists: True

PUNCTUATION MODEL
Directory exists: True
Files: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']

TRANSLATION MODEL
Directory exists: True
Epoch 5 checkpoint: True
Epoch 5 backup: True

INPUT ZIP
ZIP exists: True
Files inside ZIP:
  Sample-Raw-No-Punctuations/
  Sample-Raw-No-Punctuations/sample-1.txt
  Sample-Raw-No-Punctuations/sample-2.txt
  Sample-Raw-No-Punctuations/sample-3.txt
  Sample-Raw-No-Punctuations/sample-4.txt
  Sample-Raw-No-Punctuations/sample-5.txt
  Sample-Raw-No-Punctuations/sample-6.txt
  Sample-Raw-No-Punctuations/sample-7.txt

GPU
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU memory: 4.0 GB


In [3]:

from pathlib import Path
import zipfile

# Paths
ZIP_PATH = DATA_DIR / "Sample-Raw-No-Punctuations.zip"
EXTRACT_DIR = DATA_DIR / "extracted_input"

# Extract ZIP safely
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for member in z.infolist():
        target = (EXTRACT_DIR / member.filename).resolve()

        # Prevent paths outside the extraction directory
        if not target.is_relative_to(EXTRACT_DIR.resolve()):
            raise ValueError(f"Unsafe ZIP path: {member.filename}")

    z.extractall(EXTRACT_DIR)

print("Extraction completed.")
print("Extracted to:", EXTRACT_DIR)

# Locate the seven text files
INPUT_DIR = EXTRACT_DIR / "Sample-Raw-No-Punctuations"

input_files = sorted(INPUT_DIR.glob("sample-*.txt"))

if len(input_files) != 7:
    raise ValueError(
        f"Expected 7 input files, but found {len(input_files)}."
    )

# Read and preserve original text
raw_texts = {}

for file_path in input_files:
    text = file_path.read_text(
        encoding="utf-8-sig",
        errors="replace"
    )

    raw_texts[file_path.name] = text

    print("\n" + "=" * 60)
    print("FILE:", file_path.name)
    print("Characters:", len(text))
    print("Lines:", len(text.splitlines()))
    print("Preview:")
    print(text[:500])


Extraction completed.
Extracted to: E:\punctuation_project\data\extracted_input

FILE: sample-1.txt
Characters: 3797
Lines: 1
Preview:
भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है

FILE: sample-2.txt
Characters: 3825
Lines: 1
Preview:
प्रमुख स्थानों पर क्लोज सर्किट टीवी कैमरे भी लगाए गए हैं गंगा तट के रेतीले मैदान में मेला लगता है किंतु गढ़ मुक्तेश्वर के गंगा मंदिर में अब 86 सीढ़ियाँ ही बची हैं मेला स्थल दिल्ली से लगभग 90 किलोमीटर पड़ता है उस वक्त गंगा की धारा के साथ बहते दीपों का दृश्य बड़ा ही शांतिदायक और मनोहारी प्रतीत होता है अब गंगा 

In [4]:

import torch
from transformers import AutoTokenizer, AlbertForTokenClassification
from pathlib import Path

# Model path
PUNCTUATION_MODEL_DIR = Path(
    r"E:\punctuation_project\models\indicbert_punctuation_weighted_final"
)

# Select device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading punctuation tokenizer...")

punctuation_tokenizer = AutoTokenizer.from_pretrained(
    str(PUNCTUATION_MODEL_DIR)
)

print("Loading trained punctuation model...")

punctuation_model = AlbertForTokenClassification.from_pretrained(
    str(PUNCTUATION_MODEL_DIR)
)

punctuation_model.to(DEVICE)
punctuation_model.eval()

print("\nMODEL LOADED SUCCESSFULLY")
print("=" * 45)
print("Device:", DEVICE)
print("Model class:", type(punctuation_model).__name__)
print("Tokenizer class:", type(punctuation_tokenizer).__name__)
print("Number of labels:", punctuation_model.config.num_labels)
print("Label mapping:", punctuation_model.config.id2label)

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / (1024**3), 3),
        "GB"
    )


Loading punctuation tokenizer...
Loading trained punctuation model...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]


MODEL LOADED SUCCESSFULLY
Device: cuda
Model class: AlbertForTokenClassification
Tokenizer class: AlbertTokenizer
Number of labels: 5
Label mapping: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4'}
GPU memory allocated: 0.123 GB


In [5]:

# Restore the label names used during training

LABEL_NAMES = {
    0: "No punctuation",
    1: "Comma-like",
    2: "Sentence ending",
    3: "Question mark",
    4: "Exclamation mark"
}

LABEL_TO_ID = {
    name: idx for idx, name in LABEL_NAMES.items()
}

# Update the in-memory model configuration
punctuation_model.config.id2label = LABEL_NAMES
punctuation_model.config.label2id = LABEL_TO_ID

print("PUNCTUATION LABEL MAPPING")
print("=" * 40)

for label_id, label_name in LABEL_NAMES.items():
    print(f"{label_id}: {label_name}")

print("\nTokenizer maximum length:",
      punctuation_tokenizer.model_max_length)

print("Model maximum positions:",
      punctuation_model.config.max_position_embeddings)

print("Number of labels:",
      punctuation_model.config.num_labels)


PUNCTUATION LABEL MAPPING
0: No punctuation
1: Comma-like
2: Sentence ending
3: Question mark
4: Exclamation mark

Tokenizer maximum length: 1000000000000000019884624838656
Model maximum positions: 512
Number of labels: 5


In [6]:

from pathlib import Path
from datasets import load_from_disk

PUNCTUATION_DATASET_DIR = (
    DATA_DIR / "processed" / "punctuation_processed"
)

print("Dataset path:", PUNCTUATION_DATASET_DIR)
print("Exists:", PUNCTUATION_DATASET_DIR.exists())

if not PUNCTUATION_DATASET_DIR.exists():
    raise FileNotFoundError(
        "Training dataset not found at the expected path."
    )

punctuation_data = load_from_disk(str(PUNCTUATION_DATASET_DIR))

print("\nDataset structure:")
print(punctuation_data)

# Inspect one training example
example = punctuation_data["train"][0]

print("\nAvailable columns:", example.keys())
print("Input token count:", len(example["input_ids"]))
print("Label count:", len(example["labels"]))

input_ids = example["input_ids"]
labels = example["labels"]

tokens = punctuation_tokenizer.convert_ids_to_tokens(input_ids)

print("\nTOKEN-LEVEL LABEL INSPECTION")
print("=" * 65)
print(f"{'POSITION':<10} {'TOKEN':<25} {'LABEL ID':<12} {'MEANING'}")
print("-" * 65)

for i, (token, label) in enumerate(zip(tokens, labels)):
    if label == -100:
        meaning = "IGNORED"
    else:
        meaning = LABEL_NAMES.get(label, "UNKNOWN")

    print(f"{i:<10} {token:<25} {label:<12} {meaning}")

    if i >= 39:
        break

print("\nSequence length limit:", punctuation_model.config.max_position_embeddings)


Dataset path: E:\punctuation_project\data\processed\punctuation_processed
Exists: True

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2251236
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 278086
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 283654
    })
})

Available columns: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input token count: 60
Label count: 60

TOKEN-LEVEL LABEL INSPECTION
POSITION   TOKEN                     LABEL ID     MEANING
-----------------------------------------------------------------
0          [CLS]                     -100         IGNORED
1          ▁सत                       -100         IGNORED
2          र                         0            No punctuation
3          ▁हद                       -100         IGNORED
4          यम            

In [7]:

example = punctuation_data["train"][0]

tokens = punctuation_tokenizer.convert_ids_to_tokens(
    example["input_ids"]
)

labels = example["labels"]

print("LABELED TOKENS ONLY")
print("=" * 65)

for i, (token, label) in enumerate(zip(tokens, labels)):
    if label != -100:
        print(
            f"Position: {i:3} | "
            f"Token: {token:20} | "
            f"Label: {label} ({LABEL_NAMES[label]})"
        )

print("\nTOTAL TOKENS:", len(tokens))
print("LABELED TOKENS:", sum(x != -100 for x in labels))
print("IGNORED TOKENS:", sum(x == -100 for x in labels))


LABELED TOKENS ONLY
Position:   2 | Token: र                    | Label: 0 (No punctuation)
Position:   4 | Token: यम                   | Label: 0 (No punctuation)
Position:   7 | Token: म                    | Label: 1 (Comma-like)
Position:  11 | Token: യ                    | Label: 0 (No punctuation)
Position:  12 | Token: ▁भरत                 | Label: 0 (No punctuation)
Position:  13 | Token: ▁स                   | Label: 0 (No punctuation)
Position:  16 | Token: म                    | Label: 0 (No punctuation)
Position:  18 | Token: ष                    | Label: 0 (No punctuation)
Position:  19 | Token: ▁क                   | Label: 0 (No punctuation)
Position:  21 | Token: धक                   | Label: 0 (No punctuation)
Position:  23 | Token: भवन                  | Label: 0 (No punctuation)
Position:  27 | Token: य                    | Label: 0 (No punctuation)
Position:  28 | Token: ▁म                   | Label: 0 (No punctuation)
Position:  29 | Token: ▁स                   | La

In [8]:

import json
from pathlib import Path

notebook_files = list(PROJECT_DIR.glob("*.ipynb"))

search_terms = [
    "word_ids",
    "punctuation_processed",
    "labels",
    "AutoTokenizer",
    "is_split_into_words",
    "-100",
    "map("
]

print("SEARCHING NOTEBOOKS")
print("=" * 65)

for notebook_path in notebook_files:
    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    for cell_number, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        matched_terms = [
            term for term in search_terms
            if term in source
        ]

        # Find cells likely related to dataset preprocessing
        if (
            "punctuation_processed" in source
            or "word_ids" in source
            or ("-100" in source and "labels" in source)
        ):
            print("\n" + "=" * 65)
            print("NOTEBOOK:", notebook_path.name)
            print("CELL:", cell_number)
            print("MATCHES:", matched_terms)
            print("-" * 65)
            print(source[:7000])


SEARCHING NOTEBOOKS

NOTEBOOK: 04_end_to_end_pipeline.ipynb
CELL: 4
MATCHES: ['punctuation_processed', 'labels', '-100']
-----------------------------------------------------------------

from pathlib import Path
from datasets import load_from_disk

PUNCTUATION_DATASET_DIR = (
    DATA_DIR / "processed" / "punctuation_processed"
)

print("Dataset path:", PUNCTUATION_DATASET_DIR)
print("Exists:", PUNCTUATION_DATASET_DIR.exists())

if not PUNCTUATION_DATASET_DIR.exists():
    raise FileNotFoundError(
        "Training dataset not found at the expected path."
    )

punctuation_data = load_from_disk(str(PUNCTUATION_DATASET_DIR))

print("\nDataset structure:")
print(punctuation_data)

# Inspect one training example
example = punctuation_data["train"][0]

print("\nAvailable columns:", example.keys())
print("Input token count:", len(example["input_ids"]))
print("Label count:", len(example["labels"]))

input_ids = example["input_ids"]
labels = example["labels"]

tokens = punctuation_tokenizer

In [9]:

import torch
import re

# Same sequence length used during punctuation training
PUNCTUATION_MAX_LENGTH = 64
MAX_WORD_TOKENS = PUNCTUATION_MAX_LENGTH - 2

# Punctuation symbols
PUNCTUATION_SYMBOLS = {
    0: "",
    1: ",",
    2: "।",
    3: "?",
    4: "!"
}


def restore_punctuation(text):
    """
    Restore punctuation in Hindi text using the trained IndicBERT model.

    Uses the same final-subword label alignment as training.
    """

    if not isinstance(text, str) or not text.strip():
        return ""

    # Normalize whitespace without changing the original words
    words = text.split()

    # Tokenize each original word independently
    word_data = []

    for word in words:
        pieces = punctuation_tokenizer.tokenize(word)

        if not pieces:
            pieces = [punctuation_tokenizer.unk_token]

        # Keep original word and its subword tokens
        word_data.append({
            "word": word,
            "pieces": pieces
        })

    # Build chunks without splitting individual words
    chunks = []
    current_chunk = []
    current_length = 0

    for item in word_data:
        piece_count = len(item["pieces"])

        # Handle an unusually long individual word
        if piece_count > MAX_WORD_TOKENS:
            raise ValueError(
                f"Word exceeds model capacity: {item['word']}"
            )

        if current_chunk and current_length + piece_count > MAX_WORD_TOKENS:
            chunks.append(current_chunk)
            current_chunk = []
            current_length = 0

        current_chunk.append(item)
        current_length += piece_count

    if current_chunk:
        chunks.append(current_chunk)

    restored_words = []

    punctuation_model.eval()

    with torch.inference_mode():

        for chunk in chunks:

            tokens = [punctuation_tokenizer.cls_token]
            word_positions = []

            # Reproduce training token construction
            for item in chunk:
                pieces = item["pieces"]

                tokens.extend(pieces)

                # Position of the final subword for this word
                word_positions.append(len(tokens) - 1)

            tokens.append(punctuation_tokenizer.sep_token)

            input_ids = punctuation_tokenizer.convert_tokens_to_ids(
                tokens
            )

            input_tensor = torch.tensor(
                [input_ids],
                dtype=torch.long,
                device=DEVICE
            )

            attention_mask = torch.ones_like(input_tensor)

            outputs = punctuation_model(
                input_ids=input_tensor,
                attention_mask=attention_mask
            )

            predictions = outputs.logits.argmax(dim=-1)[0]

            # Retrieve the predicted label for each word's final subword
            for item, position in zip(chunk, word_positions):

                label_id = int(predictions[position].item())

                punctuation = PUNCTUATION_SYMBOLS.get(
                    label_id, ""
                )

                restored_words.append(
                    item["word"] + punctuation
                )

            del input_tensor, attention_mask, outputs, predictions

    return " ".join(restored_words)


print("Punctuation restoration function created.")
print("Maximum sequence length:", PUNCTUATION_MAX_LENGTH)
print("Maximum word subword tokens per chunk:", MAX_WORD_TOKENS)


Punctuation restoration function created.
Maximum sequence length: 64
Maximum word subword tokens per chunk: 62


In [10]:

# Test on a short portion of sample-1
test_text = raw_texts["sample-1.txt"][:500]

restored_test = restore_punctuation(test_text)

print("ORIGINAL TEXT")
print("=" * 70)
print(test_text)

print("\nRESTORED TEXT")
print("=" * 70)
print(restored_test)

print("\nGPU MEMORY")
if torch.cuda.is_available():
    print(
        round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
        "GB allocated"
    )


ORIGINAL TEXT
भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है

RESTORED TEXT
भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था, चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है, द्वारका चार धामों में एक धाम भी है कहते। हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज। युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई अयोध्या भगवान श

In [11]:

import torch

def inspect_punctuation_predictions(text):
    words = text.split()
    results = []
    chunk = []
    chunk_length = 0

    def process_chunk(items):
        tokens = [punctuation_tokenizer.cls_token]
        positions = []

        for word in items:
            pieces = punctuation_tokenizer.tokenize(word)
            tokens.extend(pieces)
            positions.append(len(tokens) - 1)

        tokens.append(punctuation_tokenizer.sep_token)

        input_ids = punctuation_tokenizer.convert_tokens_to_ids(tokens)
        input_tensor = torch.tensor(
            [input_ids], dtype=torch.long, device=DEVICE
        )

        with torch.inference_mode():
            logits = punctuation_model(input_ids=input_tensor).logits[0]
            probabilities = torch.softmax(logits, dim=-1)

        for word, position in zip(items, positions):
            probs = probabilities[position]
            label = int(probs.argmax().item())
            confidence = float(probs[label].item())

            if label != 0:
                results.append({
                    "word": word,
                    "prediction": PUNCTUATION_SYMBOLS[label],
                    "confidence": round(confidence, 3)
                })

        del input_tensor, logits, probabilities

    for word in words:
        piece_count = len(punctuation_tokenizer.tokenize(word))

        if chunk and chunk_length + piece_count > MAX_WORD_TOKENS:
            process_chunk(chunk)
            chunk = []
            chunk_length = 0

        chunk.append(word)
        chunk_length += piece_count

    if chunk:
        process_chunk(chunk)

    for item in results:
        print(
            f"{item['word']:<20} "
            f"{item['prediction']:<5} "
            f"confidence={item['confidence']}"
        )

    print("\nTotal punctuation predictions:", len(results))


inspect_punctuation_predictions(test_text)


था                   ,     confidence=0.834
है                   ,     confidence=0.674
कहते                 ।     confidence=0.952
धर्मराज              ।     confidence=0.925
है                   ।     confidence=0.964

Total punctuation predictions: 5


In [14]:

from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import torch

validation_dataset = dataset["validation"].select(range(1000))

# Convert dataset columns into PyTorch tensors.
validation_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=1,
    shuffle=False
)

all_true = []
all_pred = []

punctuation_model.eval()

with torch.inference_mode():
    for batch in validation_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = punctuation_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits

        predictions = logits.argmax(dim=-1)
        valid_positions = labels != -100

        all_true.extend(labels[valid_positions].cpu().tolist())
        all_pred.extend(predictions[valid_positions].cpu().tolist())

print("VALIDATION RESULTS (1,000 examples)")
print("=" * 65)

print(classification_report(
    all_true,
    all_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=[
        "No punctuation",
        "Comma",
        "Sentence ending",
        "Question mark",
        "Exclamation mark"
    ],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    all_true,
    all_pred,
    labels=[0, 1, 2, 3, 4]
))


VALIDATION RESULTS (1,000 examples)
                  precision    recall  f1-score   support

  No punctuation       0.98      0.97      0.97     13927
           Comma       0.33      0.38      0.35       510
 Sentence ending       0.92      0.96      0.94       738
   Question mark       0.00      0.00      0.00         1
Exclamation mark       0.00      0.00      0.00         2

        accuracy                           0.95     15178
       macro avg       0.44      0.46      0.45     15178
    weighted avg       0.95      0.95      0.95     15178

Confusion Matrix:
[[13503   377    47     0     0]
 [  300   194    16     0     0]
 [   15    16   707     0     0]
 [    0     0     1     0     0]
 [    1     0     1     0     0]]


In [16]:

punctuation_evaluation = {
    "validation_examples": 278086,
    "validation_tokens": 4363645,
    "accuracy": 0.9409,
    "macro_f1": 0.4570,
    "class_f1": {
        "no_punctuation": 0.9686,
        "comma": 0.3984,
        "sentence_ending": 0.9180,
        "question_mark": 0.0,
        "exclamation_mark": 0.0
    },
    "peak_gpu_memory_gb": 0.39
}

print("Punctuation model evaluation recorded.")
print(punctuation_evaluation)


Punctuation model evaluation recorded.
{'validation_examples': 278086, 'validation_tokens': 4363645, 'accuracy': 0.9409, 'macro_f1': 0.457, 'class_f1': {'no_punctuation': 0.9686, 'comma': 0.3984, 'sentence_ending': 0.918, 'question_mark': 0.0, 'exclamation_mark': 0.0}, 'peak_gpu_memory_gb': 0.39}


In [17]:

import json

TRANSLATION_NOTEBOOK = PROJECT_DIR / "03_translation.ipynb"

with open(TRANSLATION_NOTEBOOK, "r", encoding="utf-8") as f:
    translation_nb = json.load(f)

search_terms = [
    "class ",
    "generate_translation",
    "vocabulary_mapping",
    "load_state_dict",
    "torch.load",
    "def translate",
    "def generate",
]

matched_cells = []

for i, cell in enumerate(translation_nb["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(term in source for term in search_terms):
        matched_cells.append((i, source))

print("Relevant translation notebook cells:", len(matched_cells))
print("=" * 80)

for cell_number, source in matched_cells:
    print(f"\n\nCELL {cell_number}")
    print("-" * 80)
    print(source[:12000])
    if len(source) > 12000:
        print("\n[Cell truncated. Remaining characters:", len(source) - 12000, "]")


Relevant translation notebook cells: 18


CELL 8
--------------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F

# Model configuration
VOCAB_SIZE = len(mt5_tokenizer)
D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
FFN_DIM = 256
DROPOUT = 0.1

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 64

PAD_ID = mt5_tokenizer.pad_token_id
EOS_ID = mt5_tokenizer.eos_token_id
START_ID = mt5_tokenizer.pad_token_id


class HindiEnglishTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        # One shared embedding matrix for Hindi and English
        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL,
            padding_idx=PAD_ID
        )

        # Positional embeddings
        self.source_position = nn.Embedding(
            MAX_SOURCE_LENGTH, D_MODEL
        )

        self.target_position = nn.Embedding(
            MAX_TARGET_LENGTH, D

In [18]:

import json
import torch

# Paths
TRANSLATION_MODEL_DIR = MODELS_DIR / "hindi_english_translation"

checkpoint_path = TRANSLATION_MODEL_DIR / "epoch5_checkpoint.pt"
backup_path = TRANSLATION_MODEL_DIR / "epoch5_final_backup.pt"
mapping_path = TRANSLATION_MODEL_DIR / "vocabulary_mapping.json"

# Verify files
for path in [checkpoint_path, backup_path, mapping_path]:
    print(path.name, ":", path.exists())

# Load checkpoint metadata and weights onto CPU
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

# Load compact vocabulary mapping
with open(mapping_path, "r", encoding="utf-8") as f:
    compact_to_original = json.load(f)

print("\nCHECKPOINT INFORMATION")
print("=" * 55)
print("Epoch:", checkpoint["epoch"])
print("Training steps:", checkpoint["step"])
print("Vocabulary size:", checkpoint["vocab_size"])
print("Model dimension:", checkpoint["d_model"])

print("\nEMBEDDING SHAPE")
print(checkpoint["model_state_dict"]["embedding.weight"].shape)

print("\nVOCABULARY MAPPING")
print("Mapping entries:", len(compact_to_original))

print("\nCHECKPOINT SIZE")
print(round(checkpoint_path.stat().st_size / (1024 ** 2), 2), "MB")

assert checkpoint["epoch"] == 5
assert checkpoint["vocab_size"] == 32000
assert len(compact_to_original) == 32000
assert checkpoint["model_state_dict"]["embedding.weight"].shape == (32000, 128)

print("\nEpoch 5 checkpoint and vocabulary mapping verified.")


epoch5_checkpoint.pt : True
epoch5_final_backup.pt : True
vocabulary_mapping.json : True

CHECKPOINT INFORMATION
Epoch: 5
Training steps: 87500
Vocabulary size: 32000
Model dimension: 128

EMBEDDING SHAPE
torch.Size([32000, 128])

VOCABULARY MAPPING
Mapping entries: 32000

CHECKPOINT SIZE
54.91 MB

Epoch 5 checkpoint and vocabulary mapping verified.


In [19]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import gc

# Exact configuration from translation notebook Cell 15
VOCAB_SIZE = 32000
D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
FFN_DIM = 256
DROPOUT = 0.1

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

# Compact vocabulary special-token IDs
PAD_ID = 0
EOS_ID = 1
UNK_ID = 2
START_ID = PAD_ID


class HindiEnglishTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL,
            padding_idx=PAD_ID
        )

        self.source_position = nn.Embedding(
            MAX_SOURCE_LENGTH, D_MODEL
        )

        self.target_position = nn.Embedding(
            MAX_TARGET_LENGTH, D_MODEL
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NUM_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=NUM_ENCODER_LAYERS
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=D_MODEL,
            nhead=NUM_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu"
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=NUM_DECODER_LAYERS
        )

        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, source_ids, decoder_ids):

        source_length = source_ids.size(1)
        target_length = decoder_ids.size(1)

        source_positions = torch.arange(
            source_length, device=source_ids.device
        ).unsqueeze(0)

        target_positions = torch.arange(
            target_length, device=decoder_ids.device
        ).unsqueeze(0)

        source = self.embedding(source_ids)
        source = source + self.source_position(source_positions)
        source = self.dropout(source)

        source_padding_mask = source_ids.eq(PAD_ID)

        memory = self.encoder(
            source,
            src_key_padding_mask=source_padding_mask
        )

        target = self.embedding(decoder_ids)
        target = target + self.target_position(target_positions)
        target = self.dropout(target)

        target_padding_mask = decoder_ids.eq(PAD_ID).clone()
        target_padding_mask[:, 0] = False

        causal_mask = torch.triu(
            torch.ones(
                target_length,
                target_length,
                dtype=torch.bool,
                device=decoder_ids.device
            ),
            diagonal=1
        )

        decoded = self.decoder(
            tgt=target,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask
        )

        return F.linear(decoded, self.embedding.weight)


# Load the exact trained architecture
translation_model = HindiEnglishTransformer().to(DEVICE)

# Restore Epoch 5 weights
translation_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

translation_model.eval()

parameter_count = sum(
    p.numel() for p in translation_model.parameters()
)

print("Translation model loaded successfully.")
print("Parameters:", f"{parameter_count:,}")
print("Vocabulary size:", VOCAB_SIZE)
print("Device:", DEVICE)
print("Training epoch:", checkpoint["epoch"])
print("Training steps:", checkpoint["step"])

if torch.cuda.is_available():
    print(
        "Current GPU memory:",
        round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
        "GB"
    )

assert parameter_count == 4_791_296
print("Architecture and checkpoint verification passed.")


Translation model loaded successfully.
Parameters: 4,791,296
Vocabulary size: 32000
Device: cuda
Training epoch: 5
Training steps: 87500
Current GPU memory: 0.149 GB
Architecture and checkpoint verification passed.


In [20]:

import json
from transformers import AutoTokenizer

# Load the same tokenizer used during translation training
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

# Load the compact vocabulary mapping
with open(
    TRANSLATION_MODEL_DIR / "vocabulary_mapping.json",
    "r",
    encoding="utf-8"
) as f:
    compact_to_original_list = json.load(f)

# Convert to dictionaries
compact_to_original = {
    compact_id: int(original_id)
    for compact_id, original_id in enumerate(compact_to_original_list)
}

original_to_compact = {
    original_id: compact_id
    for compact_id, original_id in compact_to_original.items()
}

# Recover special-token IDs from the original tokenizer
PAD_ID = original_to_compact[tokenizer.pad_token_id]
EOS_ID = original_to_compact[tokenizer.eos_token_id]
UNK_ID = original_to_compact[tokenizer.unk_token_id]
START_ID = PAD_ID

print("TOKENIZER AND MAPPING VERIFICATION")
print("=" * 45)
print("Tokenizer:", tokenizer.name_or_path)
print("Original vocabulary size:", len(tokenizer))
print("Compact vocabulary size:", len(compact_to_original))

print("\nSPECIAL TOKENS")
print("Original PAD ID:", tokenizer.pad_token_id)
print("Compact PAD ID:", PAD_ID)
print("Original EOS ID:", tokenizer.eos_token_id)
print("Compact EOS ID:", EOS_ID)
print("Original UNK ID:", tokenizer.unk_token_id)
print("Compact UNK ID:", UNK_ID)

assert len(compact_to_original) == VOCAB_SIZE
assert compact_to_original[PAD_ID] == tokenizer.pad_token_id
assert compact_to_original[EOS_ID] == tokenizer.eos_token_id
assert compact_to_original[UNK_ID] == tokenizer.unk_token_id

print("\nTokenizer and compact vocabulary mapping verified.")


TOKENIZER AND MAPPING VERIFICATION
Tokenizer: google/mt5-small
Original vocabulary size: 250100
Compact vocabulary size: 32000

SPECIAL TOKENS
Original PAD ID: 0
Compact PAD ID: 0
Original EOS ID: 1
Compact EOS ID: 1
Original UNK ID: 2
Compact UNK ID: 2

Tokenizer and compact vocabulary mapping verified.


In [21]:

@torch.no_grad()
def generate_translation(source_ids):
    translation_model.eval()

    # Prepare source input
    source_ids = source_ids.unsqueeze(0).to(DEVICE)

    source_length = source_ids.size(1)
    source_positions = torch.arange(
        source_length, device=DEVICE
    ).unsqueeze(0)

    source_padding_mask = source_ids.eq(PAD_ID)

    # Encode the Hindi input only once
    source = translation_model.embedding(source_ids)
    source = source + translation_model.source_position(source_positions)
    source = translation_model.dropout(source)

    memory = translation_model.encoder(
        source,
        src_key_padding_mask=source_padding_mask
    )

    # Begin decoding with the START token
    generated = [START_ID]

    for _ in range(MAX_TARGET_LENGTH - 1):

        decoder_ids = torch.tensor(
            [generated],
            dtype=torch.long,
            device=DEVICE
        )

        target_length = decoder_ids.size(1)

        target_positions = torch.arange(
            target_length, device=DEVICE
        ).unsqueeze(0)

        target = translation_model.embedding(decoder_ids)
        target = target + translation_model.target_position(target_positions)
        target = translation_model.dropout(target)

        target_padding_mask = decoder_ids.eq(PAD_ID).clone()
        target_padding_mask[:, 0] = False

        causal_mask = torch.triu(
            torch.ones(
                target_length,
                target_length,
                dtype=torch.bool,
                device=DEVICE
            ),
            diagonal=1
        )

        decoded = translation_model.decoder(
            tgt=target,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask
        )

        logits = F.linear(
            decoded[:, -1, :],
            translation_model.embedding.weight
        )

        next_token = logits.argmax(dim=-1).item()

        if next_token == EOS_ID:
            break

        generated.append(next_token)

    # Exclude the START token
    return generated[1:]


def translate_hindi(text):
    # Tokenize using the original mT5 tokenizer
    original_ids = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        add_special_tokens=True
    )["input_ids"]

    # Convert original mT5 IDs into compact vocabulary IDs
    compact_ids = [
        original_to_compact.get(token_id, UNK_ID)
        for token_id in original_ids
    ]

    source_tensor = torch.tensor(
        compact_ids,
        dtype=torch.long
    )

    # Generate compact English token IDs
    generated_ids = generate_translation(source_tensor)

    # Convert compact IDs back to original mT5 IDs
    original_generated_ids = [
        compact_to_original[token_id]
        for token_id in generated_ids
    ]

    # Decode into readable English
    return tokenizer.decode(
        original_generated_ids,
        skip_special_tokens=True
    )


# Test on one Hindi sentence
test_hindi = "भारत एक विविधताओं वाला देश है।"

test_translation = translate_hindi(test_hindi)

print("HINDI INPUT:")
print(test_hindi)

print("\nMODEL TRANSLATION:")
print(test_translation)

if torch.cuda.is_available():
    print(
        "\nGPU memory:",
        round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
        "GB"
    )


e:\punctuation_project\.venv\Lib\site-packages\torch\nn\modules\transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(


HINDI INPUT:
भारत एक विविधताओं वाला देश है।

MODEL TRANSLATION:
The country is a country.

GPU memory: 0.149 GB


In [22]:

def translate_hindi(text):
    # Convert Hindi text into original mT5 token IDs
    original_ids = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        add_special_tokens=True
    )["input_ids"]

    # Convert original IDs into compact vocabulary IDs
    compact_ids = [
        original_to_compact.get(token_id, UNK_ID)
        for token_id in original_ids
    ]

    source_tensor = torch.tensor(
        compact_ids,
        dtype=torch.long,
        device=DEVICE
    )

    # Generate compact English token IDs
    generated_ids = generate_translation(source_tensor)

    # Convert compact IDs back to original mT5 IDs
    original_generated_ids = [
        compact_to_original[token_id]
        for token_id in generated_ids
    ]

    return tokenizer.decode(
        original_generated_ids,
        skip_special_tokens=True
    )


test_sentences = [
    "भारत एक विविधताओं वाला देश है।",
    "भारत की राजधानी नई दिल्ली है।",
    "कपास से कपड़ा बनाया जाता है।",
    "रेशम का उत्पादन रेशम के कीड़ों से होता है।",
    "हमें अपने पर्यावरण की रक्षा करनी चाहिए।"
]

for i, sentence in enumerate(test_sentences, 1):
    translation = translate_hindi(sentence)

    print(f"\nTest {i}")
    print("Hindi :", sentence)
    print("English:", translation)

if torch.cuda.is_available():
    print(
        "\nGPU memory:",
        round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
        "GB"
    )



Test 1
Hindi : भारत एक विविधताओं वाला देश है।
English: The country is a country.

Test 2
Hindi : भारत की राजधानी नई दिल्ली है।
English: Delhi is Delhi.

Test 3
Hindi : कपास से कपड़ा बनाया जाता है।
English: It is created.

Test 4
Hindi : रेशम का उत्पादन रेशम के कीड़ों से होता है।
English: The trees of silk is a female.

Test 5
Hindi : हमें अपने पर्यावरण की रक्षा करनी चाहिए।
English: The security should be able to security.

GPU memory: 0.149 GB


In [23]:

import json
from pathlib import Path

# Create a separate output directory
OUTPUT_DIR = DATA_DIR / "pipeline_outputs"
PUNCTUATION_OUTPUT_DIR = OUTPUT_DIR / "01_punctuation"

PUNCTUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

punctuated_texts = {}

for filename, raw_text in raw_texts.items():

    print(f"\nProcessing: {filename}")

    # Restore punctuation using the existing trained model
    punctuated_text = restore_punctuation(raw_text)

    punctuated_texts[filename] = punctuated_text

    # Save output without changing the original file
    output_path = PUNCTUATION_OUTPUT_DIR / filename

    output_path.write_text(
        punctuated_text,
        encoding="utf-8"
    )

    print("Input characters :", len(raw_text))
    print("Output characters:", len(punctuated_text))
    print("Saved to:", output_path)

print("\n" + "=" * 55)
print("PUNCTUATION RESTORATION COMPLETED")
print("=" * 55)

print("Files processed:", len(punctuated_texts))
print("Output directory:", PUNCTUATION_OUTPUT_DIR)



Processing: sample-1.txt
Input characters : 3797
Output characters: 3880
Saved to: E:\punctuation_project\data\pipeline_outputs\01_punctuation\sample-1.txt

Processing: sample-2.txt
Input characters : 3825
Output characters: 3905
Saved to: E:\punctuation_project\data\pipeline_outputs\01_punctuation\sample-2.txt

Processing: sample-3.txt
Input characters : 3931
Output characters: 4016
Saved to: E:\punctuation_project\data\pipeline_outputs\01_punctuation\sample-3.txt

Processing: sample-4.txt
Input characters : 5428
Output characters: 5551
Saved to: E:\punctuation_project\data\pipeline_outputs\01_punctuation\sample-4.txt

Processing: sample-5.txt
Input characters : 4992
Output characters: 5107
Saved to: E:\punctuation_project\data\pipeline_outputs\01_punctuation\sample-5.txt

Processing: sample-6.txt
Input characters : 2330
Output characters: 2399
Saved to: E:\punctuation_project\data\pipeline_outputs\01_punctuation\sample-6.txt

Processing: sample-7.txt
Input characters : 3913
Output c

In [24]:

for filename, text in punctuated_texts.items():

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    print(text[:700])



sample-1.txt
भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था, चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है, द्वारका चार धामों में एक धाम भी है कहते। हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज। युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई, अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है, स्टेशन का नाम कांजीवरम जरूर है, किंतु नगर का नाम। कांचीपुरम है, यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र नगर मथुरा है, दक्षिण रेलवे के चेन्नै स्टेशन से धनुष्कोटि जाने वाली मुख्य ल

sample-2.txt
प्रमुख स्थानों पर क्लोज सर्किट टीवी कैमरे भी लगाए गए हैं, गंगा तट के रेतीले मैदान में मेला लगता है, किंतु गढ़ मुक्तेश्वर के गंगा मंदिर में अब 86 सीढ़ियाँ ही बची हैं, मेला स्थल दिल्ली से लगभग 90 किलोमीटर पड़ता है उस। वक्त गंगा की धारा के साथ बहते दीपों का दृश्य बड़ा ही, श

In [25]:

import re

sentence_data = {}

for filename, text in punctuated_texts.items():

    # Split only at sentence-ending punctuation.
    sentences = re.split(r'(?<=[।?!])\s+', text.strip())

    # Remove empty entries but preserve NULL placeholders.
    sentences = [s.strip() for s in sentences if s.strip()]

    sentence_data[filename] = sentences

    print(f"\n{filename}")
    print("Sentence count:", len(sentences))

    for i, sentence in enumerate(sentences[:5], 1):
        print(f"{i}. {sentence}")



sample-1.txt
Sentence count: 20
1. भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था, चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है, द्वारका चार धामों में एक धाम भी है कहते।
2. हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज।
3. युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई, अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है, स्टेशन का नाम कांजीवरम जरूर है, किंतु नगर का नाम।
4. कांचीपुरम है, यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र नगर मथुरा है, दक्षिण रेलवे के चेन्नै स्टेशन से धनुष्कोटि जाने वाली मुख्य लाइन पर चेन्नै से 35 मील की दूरी पर चेंगलपट स्टेशन है ये।
5. सातों धर्मस्थल, सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित हैं, ऐसा कहा गया है कि चतुर्मास में इन सप्तपुरियों का दर्शन मोक्ष प्रदान करने वाला होता है, हिंदू धर्म में तीर्थ का बड़ा महत

In [26]:

import re

for filename, sentences in sentence_data.items():

    suspicious = []

    for i in range(len(sentences) - 1):
        current = sentences[i]
        following = sentences[i + 1]

        # Sentence ends with a short incomplete fragment
        last_word = current.rstrip("।?!, ").split()[-1]

        # Next sentence starts with a likely continuation
        first_word = following.split()[0]

        if (
            len(last_word) <= 5
            or first_word in {
                "हैं", "है", "था", "थी", "थे",
                "किया", "कर", "बनाता", "होता",
                "युधिष्ठिर", "कांचीपुरम", "बनाने"
            }
        ):
            suspicious.append((i + 1, current[-70:], following[:70]))

    print(f"\n{filename}: {len(suspicious)} suspicious boundaries")

    for index, left, right in suspicious[:5]:
        print(f"\nBoundary {index}:")
        print("Before:", left)
        print("After :", right)



sample-1.txt: 13 suspicious boundaries

Boundary 1:
Before: ी लाइन पर कांजीवरम स्टेशन है, द्वारका चार धामों में एक धाम भी है कहते।
After : हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नही

Boundary 2:
Before: ाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज।
After : युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भार

Boundary 3:
Before: र की पवित्र भूमि है, स्टेशन का नाम कांजीवरम जरूर है, किंतु नगर का नाम।
After : कांचीपुरम है, यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र न

Boundary 4:
Before: े वाली मुख्य लाइन पर चेन्नै से 35 मील की दूरी पर चेंगलपट स्टेशन है ये।
After : सातों धर्मस्थल, सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित

Boundary 5:
Before: मोक्ष प्रदान करने वाला होता है, हिंदू धर्म में तीर्थ का बड़ा महत्व है।
After : पूर्वोत्तर रेलवे की एक लाइन भटनी से NULL तथा दूसरी छपरा से इलाहाबाद सि

sample-2.txt: 17 suspicious boundaries

Boundary 1:
Before: सीढ़ियाँ ही बची हैं, मेला स्थल दिल्ली से लगभ

In [27]:

import re

def correct_obvious_boundaries(text):

    # Remove sentence-ending punctuation when it incorrectly
    # separates common continuation words from the previous phrase.
    continuation_words = (
        r"हैं|है|था|थी|थे|होगा|होगी|होंगे|"
        r"किया|किये|किए|कर|करने|बनाता|बनाने|"
        r"वक्त|आदि|पूर्व|पश्चात|"
        r"युधिष्ठिर|कांचीपुरम|"
        r"सोमयश|रूप"
    )

    pattern = rf"[।?!]\s+(?={continuation_words}\b)"

    corrected = re.sub(pattern, " ", text)

    # Fix obvious decimal/thousands formatting issue:
    # 7। 000 -> 7,000
    corrected = re.sub(
        r"(\d+)।\s+(\d{3})",
        r"\1,\2",
        corrected
    )

    return corrected


corrected_texts = {}

for filename, text in punctuated_texts.items():

    corrected = correct_obvious_boundaries(text)
    corrected_texts[filename] = corrected

    print(
        filename,
        "| Original:", len(text),
        "| Corrected:", len(corrected)
    )

print("\nCorrection layer applied.")


sample-1.txt | Original: 3880 | Corrected: 3876
sample-2.txt | Original: 3905 | Corrected: 3902
sample-3.txt | Original: 4016 | Corrected: 4013
sample-4.txt | Original: 5551 | Corrected: 5545
sample-5.txt | Original: 5107 | Corrected: 5105
sample-6.txt | Original: 2399 | Corrected: 2399
sample-7.txt | Original: 4008 | Corrected: 4006

Correction layer applied.


In [28]:

corrected_sentence_data = {}

for filename, text in corrected_texts.items():

    sentences = re.split(r'(?<=[।?!])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    corrected_sentence_data[filename] = sentences

    print(f"\n{'=' * 65}")
    print(filename)
    print("Sentence count:", len(sentences))

    for i, sentence in enumerate(sentences[:8], 1):
        print(f"{i}. {sentence}")



sample-1.txt
Sentence count: 16
1. भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था, चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है, द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई, अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है, स्टेशन का नाम कांजीवरम जरूर है, किंतु नगर का नाम कांचीपुरम है, यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र नगर मथुरा है, दक्षिण रेलवे के चेन्नै स्टेशन से धनुष्कोटि जाने वाली मुख्य लाइन पर चेन्नै से 35 मील की दूरी पर चेंगलपट स्टेशन है ये।
2. सातों धर्मस्थल, सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित हैं, ऐसा कहा गया है कि चतुर्मास में इन सप्तपुरियों का दर्शन मोक्ष प्रदान करने वाला होता है, हिंदू धर्म में तीर्थ का बड़ा महत्व है पूर्वो

In [29]:

CORRECTED_OUTPUT_DIR = OUTPUT_DIR / "02_corrected_hindi"

CORRECTED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for filename, text in corrected_texts.items():

    output_path = CORRECTED_OUTPUT_DIR / filename

    output_path.write_text(
        text,
        encoding="utf-8"
    )

print("Corrected Hindi files saved.")
print("Directory:", CORRECTED_OUTPUT_DIR)
print("Files:", len(corrected_texts))


Corrected Hindi files saved.
Directory: E:\punctuation_project\data\pipeline_outputs\02_corrected_hindi
Files: 7


In [31]:

from pathlib import Path
import torch

PROJECT_DIR = Path(r"E:\punctuation_project")
MODELS_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "data" / "pipeline_outputs"

ORDERING_MODEL_PATH = (
    MODELS_DIR / "sentence_ordering_indicbert" / "best_model.pt"
)

print("Checkpoint path:", ORDERING_MODEL_PATH)
print("Checkpoint exists:", ORDERING_MODEL_PATH.exists())

assert ORDERING_MODEL_PATH.exists(), (
    "Ordering checkpoint not found. Check the model directory."
)


Checkpoint path: E:\punctuation_project\models\sentence_ordering_indicbert\best_model.pt
Checkpoint exists: True


In [32]:

ordering_checkpoint = torch.load(
    ORDERING_MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

print("CHECKPOINT CONTENTS")
print("=" * 50)

if isinstance(ordering_checkpoint, dict):
    print("Top-level keys:", list(ordering_checkpoint.keys()))

    state_dict = ordering_checkpoint.get(
        "model_state_dict",
        ordering_checkpoint.get("state_dict", ordering_checkpoint)
    )

    print("\nSTATE DICTIONARY")
    for key, value in state_dict.items():
        if torch.is_tensor(value):
            print(key, tuple(value.shape))
else:
    print("Checkpoint type:", type(ordering_checkpoint))


CHECKPOINT CONTENTS
Top-level keys: ['ordering_head', 'hidden_size', 'epoch', 'metrics']

STATE DICTIONARY


In [33]:

print("HIDDEN SIZE:", ordering_checkpoint["hidden_size"])
print("EPOCH:", ordering_checkpoint["epoch"])
print("METRICS:", ordering_checkpoint["metrics"])

print("\nORDERING HEAD DETAILS")
print("=" * 50)

ordering_head_state = ordering_checkpoint["ordering_head"]

print("Type:", type(ordering_head_state))

for key, value in ordering_head_state.items():
    if torch.is_tensor(value):
        print(f"{key}: shape={tuple(value.shape)}")
    else:
        print(f"{key}: {type(value)}")


HIDDEN SIZE: 768
EPOCH: 2
METRICS: {'loss': 1.609369286712991, 'position_accuracy': 0.2060936973373778, 'exact_accuracy': 0.009976407145264578}

ORDERING HEAD DETAILS
Type: <class 'collections.OrderedDict'>
query_embeddings: shape=(1, 5, 768)
position_embeddings: shape=(1, 5, 768)
cross_attention.in_proj_weight: shape=(2304, 768)
cross_attention.in_proj_bias: shape=(2304,)
cross_attention.out_proj.weight: shape=(768, 768)
cross_attention.out_proj.bias: shape=(768,)
norm.weight: shape=(768,)
norm.bias: shape=(768,)
query_projection.weight: shape=(768, 768)
query_projection.bias: shape=(768,)
key_projection.weight: shape=(768, 768)
key_projection.bias: shape=(768,)


In [34]:

import json

ORDERING_NOTEBOOK_PATH = PROJECT_DIR / "02_sentence_ordering.ipynb"

with open(ORDERING_NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook_data = json.load(f)

search_terms = [
    "query_embeddings",
    "cross_attention",
    "class Ordering",
    "class Sentence"
]

for cell_index, cell in enumerate(notebook_data["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(term in source for term in search_terms):
        print(f"\n{'=' * 60}")
        print(f"CELL {cell_index}")
        print("=" * 60)
        print(source)



CELL 6

import os
import gc
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# 1. Reproducibility
# --------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# --------------------------------------------------
# 2. GPU memory limit
# --------------------------------------------------

if device.type == "cuda":
    # Limit this process to approximately 1.4 GB
    torch.cuda.set_per_process_memory_fraction(0.35, 0)

    torch.cuda.empty_cache()
    gc.collect()

    print(
        "PyTorch GPU memory limit:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            * 0.35 / (1024 ** 3),
            2
    

In [35]:

import math
import gc
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

class SentenceOrderingHead(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()

        self.query_embeddings = nn.Parameter(
            torch.randn(1, 5, hidden_size) * 0.02
        )
        self.position_embeddings = nn.Parameter(
            torch.randn(1, 5, hidden_size) * 0.02
        )
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=8,
            batch_first=True
        )
        self.norm = nn.LayerNorm(hidden_size)
        self.query_projection = nn.Linear(hidden_size, hidden_size)
        self.key_projection = nn.Linear(hidden_size, hidden_size)

    def forward(self, sentence_embeddings):
        batch_size = sentence_embeddings.size(0)

        keys = sentence_embeddings + self.position_embeddings
        queries = self.query_embeddings.expand(batch_size, -1, -1)

        attended, _ = self.cross_attention(queries, keys, keys)
        attended = self.norm(queries + attended)

        q = self.query_projection(attended)
        k = self.key_projection(keys)

        logits = torch.matmul(q, k.transpose(1, 2))
        return logits / math.sqrt(q.size(-1))


# Load the original tokenizer and encoder
ordering_tokenizer = AutoTokenizer.from_pretrained(
    "ai4bharat/indic-bert"
)

ordering_encoder = AutoModel.from_pretrained(
    "ai4bharat/indic-bert"
).to(DEVICE)

ordering_encoder.eval()

for parameter in ordering_encoder.parameters():
    parameter.requires_grad = False

# Reconstruct and load the trained head
ordering_head = SentenceOrderingHead(
    ordering_checkpoint["hidden_size"]
).to(DEVICE)

ordering_head.load_state_dict(
    ordering_checkpoint["ordering_head"]
)

ordering_head.eval()

print("Ordering encoder and head loaded.")
print("Encoder hidden size:", ordering_checkpoint["hidden_size"])
print("Checkpoint epoch:", ordering_checkpoint["epoch"])
print("GPU allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 3)
      if DEVICE.type == "cuda" else "CPU")


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ordering encoder and head loaded.
Encoder hidden size: 768
Checkpoint epoch: 2
GPU allocated: 0.288


In [38]:

def get_sentence_embedding(sentence):
    encoded = ordering_tokenizer(
        sentence,
        truncation=True,
        max_length=128,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)

    with torch.inference_mode():
        outputs = ordering_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Mean pooling over non-padding tokens
        token_embeddings = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1)

        embedding = (
            (token_embeddings * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        )

    return embedding


def predict_sentence_order(sentences):
    if len(sentences) != 5:
        raise ValueError("The ordering model requires exactly five sentences.")

    embeddings = torch.cat(
        [get_sentence_embedding(sentence) for sentence in sentences],
        dim=0
    ).unsqueeze(0)

    with torch.inference_mode():
        logits = ordering_head(embeddings)
        predicted_positions = logits.argmax(dim=-1)[0].tolist()

    # Each output position selects one input sentence.
    return [sentences[index] for index in predicted_positions], predicted_positions


# Test the first available group
test_filename = next(
    filename for filename, groups in sentence_groups.items() if groups
)

test_group = sentence_groups[test_filename][0]

ordered_test, predicted_positions = predict_sentence_order(test_group)

print("FILE:", test_filename)

print("\nORIGINAL ORDER")
for i, sentence in enumerate(test_group, 1):
    print(f"{i}. {sentence}")

print("\nMODEL-PROPOSED ORDER")
for i, sentence in enumerate(ordered_test, 1):
    print(f"{i}. {sentence}")

print("\nPredicted input positions:", predicted_positions)


FILE: sample-1.txt

ORIGINAL ORDER
1. भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था, चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है, द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई, अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है, स्टेशन का नाम कांजीवरम जरूर है, किंतु नगर का नाम कांचीपुरम है, यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र नगर मथुरा है, दक्षिण रेलवे के चेन्नै स्टेशन से धनुष्कोटि जाने वाली मुख्य लाइन पर चेन्नै से 35 मील की दूरी पर चेंगलपट स्टेशन है ये।
2. सातों धर्मस्थल, सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित हैं, ऐसा कहा गया है कि चतुर्मास में इन सप्तपुरियों का दर्शन मोक्ष प्रदान करने वाला होता है, हिंदू धर्म में तीर्थ का बड़ा महत्व है पूर्

In [37]:

# Recreate groups of five from the corrected sentence data

sentence_groups = {}
remaining_sentences = {}

for filename, sentences in corrected_sentence_data.items():
    groups = [
        sentences[i:i + 5]
        for i in range(0, len(sentences) - 4, 5)
    ]

    used_count = len(groups) * 5

    sentence_groups[filename] = groups
    remaining_sentences[filename] = sentences[used_count:]

print("GROUP COUNTS")
for filename, groups in sentence_groups.items():
    print(
        filename,
        "| Groups:", len(groups),
        "| Remaining:", len(remaining_sentences[filename])
    )


# Test the ordering model on the first group
test_filename = next(
    filename for filename, groups in sentence_groups.items() if groups
)

test_group = sentence_groups[test_filename][0]

ordered_test, predicted_positions = predict_sentence_order(test_group)

print("\nFILE:", test_filename)

print("\nORIGINAL ORDER")
for i, sentence in enumerate(test_group, 1):
    print(f"{i}. {sentence}")

print("\nMODEL-PROPOSED ORDER")
for i, sentence in enumerate(ordered_test, 1):
    print(f"{i}. {sentence}")

print("\nPredicted input positions:", predicted_positions)
print("Unique positions:", len(set(predicted_positions)))


GROUP COUNTS
sample-1.txt | Groups: 3 | Remaining: 1
sample-2.txt | Groups: 3 | Remaining: 2
sample-3.txt | Groups: 3 | Remaining: 2
sample-4.txt | Groups: 4 | Remaining: 2
sample-5.txt | Groups: 6 | Remaining: 1
sample-6.txt | Groups: 2 | Remaining: 2
sample-7.txt | Groups: 3 | Remaining: 3

FILE: sample-1.txt

ORIGINAL ORDER
1. भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था, चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है, द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई, अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है, स्टेशन का नाम कांजीवरम जरूर है, किंतु नगर का नाम कांचीपुरम है, यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र नगर मथुरा है, दक्षिण रेलवे के चेन्नै स्टेशन स

In [39]:

import json
from pathlib import Path

ORDERING_OUTPUT_DIR = OUTPUT_DIR / "03_ordering"
ORDERING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ordering_results = {}

for filename, groups in sentence_groups.items():

    ordering_results[filename] = {
        "ordering_method": "original_sequence_preserved",
        "automatic_ordering_used": False,
        "reason": (
            "The trained ordering model produced duplicate sentence "
            "indices and unreliable ordering predictions."
        ),
        "groups": [
            {
                "group_number": i + 1,
                "sentences": group
            }
            for i, group in enumerate(groups)
        ],
        "remaining_sentences": remaining_sentences[filename]
    }

output_path = ORDERING_OUTPUT_DIR / "ordering_results.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        ordering_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Ordering results saved.")
print("Location:", output_path)

for filename, result in ordering_results.items():
    print(
        filename,
        "| Groups:", len(result["groups"]),
        "| Remaining:", len(result["remaining_sentences"])
    )


Ordering results saved.
Location: E:\punctuation_project\data\pipeline_outputs\03_ordering\ordering_results.json
sample-1.txt | Groups: 3 | Remaining: 1
sample-2.txt | Groups: 3 | Remaining: 2
sample-3.txt | Groups: 3 | Remaining: 2
sample-4.txt | Groups: 4 | Remaining: 2
sample-5.txt | Groups: 6 | Remaining: 1
sample-6.txt | Groups: 2 | Remaining: 2
sample-7.txt | Groups: 3 | Remaining: 3


In [40]:

import json
import gc
import torch
from pathlib import Path

TRANSLATION_OUTPUT_DIR = OUTPUT_DIR / "04_english_translation"
TRANSLATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

translation_results = {}

for filename, result in ordering_results.items():

    print(f"\nProcessing: {filename}")

    output_path = TRANSLATION_OUTPUT_DIR / (
        Path(filename).stem + ".json"
    )

    # Reuse completed files if this cell is run again.
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            translation_results[filename] = json.load(f)

        print("Already translated. Loaded saved result.")
        continue

    translated_groups = []

    total_sentences = sum(
        len(group["sentences"])
        for group in result["groups"]
    ) + len(result["remaining_sentences"])

    completed = 0

    with torch.inference_mode():

        for group in result["groups"]:

            translated_sentences = []

            for hindi_sentence in group["sentences"]:

                english_sentence = translate_hindi(hindi_sentence)

                translated_sentences.append({
                    "hindi": hindi_sentence,
                    "english": english_sentence
                })

                completed += 1

                print(
                    f"\rTranslated {completed}/{total_sentences}",
                    end=""
                )

            translated_groups.append({
                "group_number": group["group_number"],
                "sentences": translated_sentences
            })

        translated_remaining = []

        for hindi_sentence in result["remaining_sentences"]:

            english_sentence = translate_hindi(hindi_sentence)

            translated_remaining.append({
                "hindi": hindi_sentence,
                "english": english_sentence
            })

            completed += 1

            print(
                f"\rTranslated {completed}/{total_sentences}",
                end=""
            )

    file_result = {
        "source_file": filename,
        "ordering_method": result["ordering_method"],
        "translation_method": "sentence_by_sentence",
        "groups": translated_groups,
        "remaining_sentences": translated_remaining
    }

    # Save immediately after each file.
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            file_result,
            f,
            ensure_ascii=False,
            indent=2
        )

    translation_results[filename] = file_result

    print(f"\nSaved: {output_path}")

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\nAll seven files processed.")
print("Output directory:", TRANSLATION_OUTPUT_DIR)



Processing: sample-1.txt
Translated 16/16
Saved: E:\punctuation_project\data\pipeline_outputs\04_english_translation\sample-1.json

Processing: sample-2.txt
Translated 17/17
Saved: E:\punctuation_project\data\pipeline_outputs\04_english_translation\sample-2.json

Processing: sample-3.txt
Translated 17/17
Saved: E:\punctuation_project\data\pipeline_outputs\04_english_translation\sample-3.json

Processing: sample-4.txt
Translated 22/22
Saved: E:\punctuation_project\data\pipeline_outputs\04_english_translation\sample-4.json

Processing: sample-5.txt
Translated 31/31
Saved: E:\punctuation_project\data\pipeline_outputs\04_english_translation\sample-5.json

Processing: sample-6.txt
Translated 12/12
Saved: E:\punctuation_project\data\pipeline_outputs\04_english_translation\sample-6.json

Processing: sample-7.txt
Translated 18/18
Saved: E:\punctuation_project\data\pipeline_outputs\04_english_translation\sample-7.json

All seven files processed.
Output directory: E:\punctuation_project\data\pi